In [ ]:
### UPLOAD THE BULDING 2025

In [ ]:
import socket
IPAddr= socket.gethostbyname(socket.gethostname())
IPAddr

'10.1.129.21'

In [10]:
import os
print(os.system('ipconfig'))

32512


sh: line 1: ipconfig: command not found


In [2]:
import socket
socket.gethostbyname("mobidb02.giub.unibe.ch")

'130.92.54.159'

In [15]:
import geopandas as gpd
from sqlalchemy import create_engine, text

# -------------------------
# Database connection
# -------------------------
database_name = "entwicklung"
port = 5432
server = "mobidb02.giub.unibe.ch"
username = "gespejo"
password = "Fellbach1993#"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{server}:{port}/{database_name}"
)

In [16]:
from sqlalchemy import text

try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print(" Connection to database successful")
except Exception as e:
    print(" Connection failed")
    print(e)


 Connection failed
(psycopg2.OperationalError) connection to server at "mobidb02.giub.unibe.ch" (130.92.54.159), port 5432 failed: could not initiate GSSAPI security context: Unspecified GSS failure.  Minor code may provide more information: Server not found in Kerberos database
connection to server at "mobidb02.giub.unibe.ch" (130.92.54.159), port 5432 failed: FATAL:  no pg_hba.conf entry for host "130.92.232.112", user "gespejo", database "entwicklung", SSL encryption
connection to server at "mobidb02.giub.unibe.ch" (130.92.54.159), port 5432 failed: FATAL:  no pg_hba.conf entry for host "130.92.232.112", user "gespejo", database "entwicklung", no encryption

(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [ ]:
import geopandas as gpd
from sqlalchemy import create_engine, text

# -------------------------
# Database connection
# -------------------------
database_name = "entwicklung"
port = 5432
server = "mobidb02.giub.unibe.ch"
username = "gespejo"
password = "####"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{server}:{port}/{database_name}"
)

# -------------------------
# Read shapefile
# -------------------------
gdf = gpd.read_file(
    "/storage/homefs/ge24z347/Zell_event/Data_forprocess/geo_hwr_footp_tlm2025.shp"
)

# Optional but recommended: check CRS
print(gdf.crs)

# If CRS is missing, SET it (example EPSG:2056 – adapt if needed)
# gdf = gdf.set_crs(epsg=2056)

# -------------------------
# Upload to PostGIS
# -------------------------
gdf.to_postgis(
    name="geo_hwr_footp_tlm2025",
    con=engine,
    schema="swf_modelling",
    if_exists="replace",   # use "append" if rerunning
    index=False
)



EPSG:2056


OperationalError: (psycopg2.OperationalError) connection to server at "mobidb02.giub.unibe.ch" (130.92.54.159), port 5432 failed: could not initiate GSSAPI security context: Unspecified GSS failure.  Minor code may provide more information: Server not found in Kerberos database
connection to server at "mobidb02.giub.unibe.ch" (130.92.54.159), port 5432 failed: FATAL:  no pg_hba.conf entry for host "130.92.232.112", user "gespejo", database "entwicklung", SSL encryption
connection to server at "mobidb02.giub.unibe.ch" (130.92.54.159), port 5432 failed: FATAL:  no pg_hba.conf entry for host "130.92.232.112", user "gespejo", database "entwicklung", no encryption

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [5]:
import xarray as xr
import pandas as pd
import numpy as np

path = "/home/gespejogutierrez/Lisflood_climada/Zell_netcdfiles/Zell_2m_Combiprecip.nc"
ds = xr.open_dataset(path)

THRESH = 0.01

# ------------------------------------------------------------------
# 1) Create wet mask from water_depth
# ------------------------------------------------------------------
mask = ds["water_depth"] > THRESH

# ------------------------------------------------------------------
# 2) Select variables AND apply the SAME mask
# ------------------------------------------------------------------
ds_sel = ds[["water_depth", "vel_mag", "flux_mag"]].where(mask)

# ------------------------------------------------------------------
# 3) Convert to DataFrame and drop dry cells
# ------------------------------------------------------------------
Zell_wet_cells = (
    ds_sel
    .to_dataframe()
    .dropna()
    .reset_index()
)

# ------------------------------------------------------------------
# 4) Add timestep index 0..10
# ------------------------------------------------------------------
times = list(Zell_wet_cells["REFERENCE_TS"].values)
time_to_step = {t: i for i, t in enumerate(times)}
Zell_wet_cells["timestep"] = Zell_wet_cells["REFERENCE_TS"].map(time_to_step).astype("int16")

# ------------------------------------------------------------------
# 5) Check result
# ------------------------------------------------------------------
Zell_wet_cells.head(), Zell_wet_cells.shape


(         REFERENCE_TS          y          x  water_depth   vel_mag  flux_mag  \
 0 2022-05-05 17:00:00  1257999.0  2702001.0        0.020  0.005000       0.0   
 1 2022-05-05 17:00:00  1257999.0  2702025.0        0.018  0.003162       0.0   
 2 2022-05-05 17:00:00  1257999.0  2702033.0        0.035  0.012166       0.0   
 3 2022-05-05 17:00:00  1257999.0  2702061.0        0.018  0.014036       0.0   
 4 2022-05-05 17:00:00  1257999.0  2702175.0        0.026  0.002236       0.0   
 
    spatial_ref  timestep  
 0            0    -29538  
 1            0    -29538  
 2            0    -29538  
 3            0    -29538  
 4            0    -29538  ,
 (2937119, 8))

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import box

# ----------------------------------------------------------
# Settings
# ----------------------------------------------------------
path = "/home/gespejogutierrez/Lisflood_climada/Zell_netcdfiles/Zell_2m_Combiprecip.nc"
THRESH = 0.01
out_path = "Zell_wet_cells_polygons.gpkg"
out_layer = "wet_cells"

# ----------------------------------------------------------
# 1) Open dataset
# ----------------------------------------------------------
ds = xr.open_dataset(path)

# ----------------------------------------------------------
# 2) Create wet mask from water_depth
# ----------------------------------------------------------
mask = ds["water_depth"] > THRESH

# ----------------------------------------------------------
# 3) Select variables and apply mask
# ----------------------------------------------------------
ds_sel = ds[["water_depth", "vel_mag", "flux_mag"]].where(mask)

# ----------------------------------------------------------
# 4) Convert to DataFrame and drop dry cells
#    Result: one row per (REFERENCE_TS, y, x) where water_depth > THRESH
# ----------------------------------------------------------
Zell_wet_cells = (
    ds_sel
    .to_dataframe()
    .dropna()
    .reset_index()
)

# ----------------------------------------------------------
# 5) Add timestep index (0..N-1), based on unique REFERENCE_TS order
# ----------------------------------------------------------
Zell_wet_cells["timestep"] = (
    Zell_wet_cells["REFERENCE_TS"]
    .astype("category")
    .cat.codes
    .astype("int16")
)

# ----------------------------------------------------------
# 6) Build polygons for each cell in EPSG:2056
#    Assumes x, y are cell centres in LV95 and grid is regular
# ----------------------------------------------------------
dx = float(ds.x[1] - ds.x[0])   # cell size in x (should be ~2 m)
dy = float(ds.y[1] - ds.y[0])   # cell size in y (can be negative)

half_dx = abs(dx) / 2.0
half_dy = abs(dy) / 2.0

geoms = [
    box(x - half_dx, y - half_dy, x + half_dx, y + half_dy)
    for x, y in zip(Zell_wet_cells["x"].values, Zell_wet_cells["y"].values)
]

# ----------------------------------------------------------
# 7) Create GeoDataFrame
# ----------------------------------------------------------
gdf = gpd.GeoDataFrame(
    Zell_wet_cells,
    geometry=geoms,
    crs="EPSG:2056",
)

print("Number of wet cell polygons:", len(gdf))
print(gdf.head())

# ----------------------------------------------------------
# 8) Save to GeoPackage
# ----------------------------------------------------------
gdf.to_file(out_path, layer=out_layer, driver="GPKG")
print(f"Saved to {out_path} (layer='{out_layer}')")


Number of wet cell polygons: 2937119
         REFERENCE_TS          y          x  water_depth   vel_mag  flux_mag  \
0 2022-05-05 17:00:00  1257999.0  2702001.0        0.020  0.005000       0.0   
1 2022-05-05 17:00:00  1257999.0  2702025.0        0.018  0.003162       0.0   
2 2022-05-05 17:00:00  1257999.0  2702033.0        0.035  0.012166       0.0   
3 2022-05-05 17:00:00  1257999.0  2702061.0        0.018  0.014036       0.0   
4 2022-05-05 17:00:00  1257999.0  2702175.0        0.026  0.002236       0.0   

   spatial_ref  timestep                                           geometry  
0            0         0  POLYGON ((2702002 1257998, 2702002 1258000, 27...  
1            0         0  POLYGON ((2702026 1257998, 2702026 1258000, 27...  
2            0         0  POLYGON ((2702034 1257998, 2702034 1258000, 27...  
3            0         0  POLYGON ((2702062 1257998, 2702062 1258000, 27...  
4            0         0  POLYGON ((2702176 1257998, 2702176 1258000, 27...  


In [5]:
import xarray as xr
import pandas as pd
import geopandas as gpd
from shapely.geometry import box

# ---- settings ----
path = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Zell_2m_accv14/Zell_2m_accv14_Combiprecip.nc"
THRESH = 0.01

ds = xr.open_dataset(path)

# 1) keep only wet cells and needed vars
wet = ds[["water_depth", "vel_mag", "flux_mag"]].where(ds["water_depth"] > THRESH)

# 2) xarray -> DataFrame, drop dry cells, get x/y/REFERENCE_TS as columns
df = wet.to_dataframe().dropna().reset_index()

# 3) timestep index 0..N-1
df["timestep"] = pd.factorize(df["REFERENCE_TS"])[0].astype("int16")

# 4) build square polygons around cell centres
dx = float(ds.x[1] - ds.x[0])
dy = float(ds.y[1] - ds.y[0])
hx, hy = abs(dx) / 2.0, abs(dy) / 2.0

df["geometry"] = [
    box(x - hx, y - hy, x + hx, y + hy)
    for x, y in zip(df["x"].to_numpy(), df["y"].to_numpy())
]

# 5) GeoDataFrame in EPSG:2056
Zell_wet_cells = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:2056")

# optional: save
#gdf.to_file("Zell_wet_cells_polygons.gpkg", layer="wet_cells", driver="GPKG")



In [6]:
Zell_wet_cells

,REFERENCE_TS,y,x,water_depth,vel_mag,flux_mag,spatial_ref,timestep,geometry
0,2022-05-05 17:00:00,1257999.0,2704263.0,0.081000,0.002236,0.000000,0,0,"POLYGON ((2704264 1257998, 2704264 1258000, 27..."
1,2022-05-05 17:00:00,1257999.0,2704265.0,0.037000,0.003162,0.000000,0,0,"POLYGON ((2704266 1257998, 2704266 1258000, 27..."
2,2022-05-05 17:00:00,1257999.0,2704321.0,0.052000,0.002236,0.000000,0,0,"POLYGON ((2704322 1257998, 2704322 1258000, 27..."
3,2022-05-05 17:00:00,1257999.0,2704405.0,0.014000,0.001414,0.000000,0,0,"POLYGON ((2704406 1257998, 2704406 1258000, 27..."
4,2022-05-05 17:00:00,1257999.0,2704423.0,0.025000,0.002828,0.000000,0,0,"POLYGON ((2704424 1257998, 2704424 1258000, 27..."
...,...,...,...,...,...,...,...,...,...
14044347,2022-05-05 22:00:00,1255001.0,2706991.0,265.463013,0.745874,197.881699,0,5,"POLYGON ((2706992 1255000, 2706992 1255002, 27..."
14044348,2022-05-05 22:00:00,1255001.0,2706993.0,265.433990,0.617744,163.989838,0,5,"POLYGON ((2706994 1255000, 2706994 1255002, 27..."
14044349,2022-05-05 22:00:00,1255001.0,2706995.0,265.362000,0.488271,129.537552,0,5,"POLYGON ((2706996 1255000, 2706996 1255002, 27..."
14044350,2022-05-05 22:00:00,1255001.0,2706997.0,265.079987,0.367104,97.408478,0,5,"POLYGON ((2706998 1255000, 2706998 1255002, 27..."


In [3]:
from sqlalchemy import create_engine
from geoalchemy2 import Geometry, WKTElement

database_name = "entwicklung"
port = 5432
server = "mobidb02.giub.unibe.ch"
username = "gespejo"
password = "Fellbach1993#"   # (careful with this in shared code)

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{server}:{port}/{database_name}"
)

# --- make a copy for upload, keep original GeoDataFrame as is ---
df_sql = Zell_wet_cells.copy()

# convert shapely geometry -> WKTElement, keep column name 'geometry'
df_sql["geometry"] = df_sql["geometry"].apply(
    lambda g: WKTElement(g.wkt, srid=2056)
)

# upload to PostGIS, geometry column is still called 'geometry'
df_sql.to_sql(
    "Zell_wet_cells",
    engine,
    schema="swf_modelling",
    if_exists="replace",   # or "append"
    index=False,
    dtype={"geometry": Geometry("POLYGON", srid=2056)},
)


NameError: name 'Zell_wet_cells' is not defined

In [10]:
import socket
socket.gethostbyname("mobidb02.giub.unibe.ch")

'130.92.54.159'

In [13]:
from sqlalchemy import create_engine

database_name = "entwicklung"
port = 5432
server = "mobidb02"
username = "gespejo"
password = "Fellbach1993#"

engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{server}:{port}/{database_name}")


Zell_wet_cells.to_sql(
    "Zell_wet_cells",
    engine,
    schema="swf_modelling",
    if_exists="replace",   # use "append" later if you rerun
    index=False
)


OperationalError: (psycopg2.OperationalError) could not translate host name "mobidb02" to address: Name or service not known

(Background on this error at: https://sqlalche.me/e/20/e3q8)